In [2]:
import os
import argparse
import pandas as pd
from typing import List, Dict
import pprint

from neo4j import GraphDatabase

from langchain_openai import OpenAIEmbeddings  # swap to your preferred embedder if needed
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_neo4j import Neo4jVector
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from IPython.display import Markdown, display

In [3]:
load_dotenv(dotenv_path=".env")

True

In [ ]:
# enter your Neo4j credentials here
os.environ["NEO4J_URI"] = ""
os.environ["NEO4J_USERNAME"] = ""
os.environ["NEO4J_PASSWORD"] = ""

In [ ]:
# check if the env variables are set 
os.environ.get("NEO4J_URI")

'neo4j+s://f33ef706.databases.neo4j.io'

# Building a Recruitment Co-pilot with GraphRAG

 Traditional RAG systems often retrieve disconnected chunks of text from a vector database. GraphRAG enhances this by retrieving information from a knowledge graph, allowing us to leverage not only the content of the text but also the rich relationships and structures within the data.

Our goal is to create a system that can intelligently recommend job candidates by:
1.  **Performing semantic search** on their project descriptions to find relevant experience.
2.  **Enriching this information** by traversing the knowledge graph to gather structured data like skills and work history.
3.  **Applying filters** based on specific criteria, such as company experience.
4.  **Synthesizing the combined information** with a Large Language Model (LLM) to provide a reasoned, evidence-based recommendation.

We will use Neo4j for our graph database, OpenAI for embeddings and language generation, and LangChain to orchestrate the entire process.

In [ ]:
# sample data
candidate_data = [
    {
        "candidate_id": "C001",
        "name": "Aanya Sharma",
        "title": "Lead ML Engineer",
        "company": "MediAssist AI",
        "projects": """
        - Designed Graph RAG pipeline linking patients, encounters, and clinical notes to FHIR resources.
        - Built WhatsApp triage bot with LangGraph agents across retrieval, coding, and escalation. 
        - Reduced nurse triage handle time by 32% (A/B, n=9,214 chats).
        - Deployed PHI-safe retrieval with field-level redaction and audit trails.
        - Applied RAG, Neo4j, LangGraph to deliver features end-to-end, collaborating with cross-functional teams.
        - Defined SLAs/SLOs and instrumented metrics to monitor quality, latency, and cost.
        """,
        "skills": ["Python", "Neo4j", "LangGraph", "RAG", "LLM Ops", "GCP"],
    },
    {
        "candidate_id": "C002",
        "name": "Miguel Santos",
        "title": "Senior DevOps Engineer",
        "company": "ShopX",
        "projects": """
        - Built multi-tenant EKS with IRSA and auto-scaling.
        - Developed Python automation scripts for CI/CD pipelines, reducing deployment times by 20%.
        - Wrote technical design docs and reviewed PRs, mentoring junior engineers on best practices.
        - Partnered with stakeholders to translate requirements into shippable increments with clear success criteria.
        """,
        "skills": ["AWS", "EKS", "Terraform", "Kubernetes", "CI/CD", "Python"],
    },
     {
        "candidate_id": "C003",
        "name": "Priya Nair",
        "title": "Data Analyst",
        "company": "Lex & Co",
        "projects": """
        - Prepared SQL reports and case digests for the litigation team.
        - Automated weekly reporting using Python and Pandas, saving 10 hours of manual work per month.
        - Collaborated with legal experts to identify key data points for case strategy.
        """,
        "skills": ["SQL", "Tableau", "Excel", "Legal Tech", "Python", "Pandas"],
    },
    {
        "candidate_id": "C004",
        "name": "Chen Wei",
        "title": "Frontend Engineer",
        "company": "ShopX",
        "projects": """
        - Led the development of a new checkout experience using React and TypeScript, improving conversion by 8%.
        - Built and maintained a shared component library to ensure design consistency across the application.
        - Optimized web performance, achieving a 40 percentage point reduction in Largest Contentful Paint (LCP).
        """,
        "skills": ["JavaScript", "TypeScript", "React", "Node.js", "Jest"],
    },
    {
        "candidate_id": "C005",
        "name": "Sofia Rossi",
        "title": "AI Product Manager",
        "company": "MediAssist AI",
        "projects": """
        - [cite_start]Defined the product roadmap for an AI-assisted help center solution.
        - [cite_start]Shipped features that improved ticket resolution by 24% via RAG-based suggestions.
        - [cite_start]Established guardrails and feedback mechanisms for AI features to ensure user safety and trust.
        """,
        "skills": ["Product Management", "Agile", "Jira", "AI Strategy", "RAG"],
    },
    {
        "candidate_id": "C006",
        "name": "David Kim",
        "title": "Cloud Engineer",
        "company": "CloudVantage",
        "projects": """
        - Deployed and managed production infrastructure on AWS using Terraform and Ansible.
        - Implemented a monitoring and alerting system with Prometheus and Grafana for critical services.
        - Assisted in achieving SOC 2 Type II compliance by implementing security best practices.
        """,
        "skills": ["AWS", "Terraform", "Ansible", "Docker", "Prometheus"],
    },
]

#### Neo4j Database Setup

The following functions handle the connection to and configuration of our Neo4j database.

- **`get_neo4j_driver()`**: Establishes and returns a connection instance (a "driver") to the Neo4j database using the credentials we set earlier.
- **`setup_neo4j_constraints()`**: Creates unique constraints on our node labels (`Candidate`, `Company`, `Role`, `Skill`). This is a crucial data integrity step. It ensures that we don't accidentally create duplicate nodes for the same entity (e.g., two nodes for the skill "Python").
- **`create_vector_index()`**: This is the core of our semantic search capability. It creates a vector index named `project_chunks` on the `Chunk` nodes. This index is specifically for the `embedding` property of the chunks. When we later search for project descriptions, Neo4j will use this index to perform a highly efficient similarity search (using the 'cosine' similarity metric) to find the most relevant text chunks.

In [7]:
# --- 2. NEO4J CONNECTION & SETUP ---

def get_neo4j_driver():
    """Initializes and returns the Neo4j driver."""
    uri = os.getenv("NEO4J_URI")
    user = os.getenv("NEO4J_USERNAME")
    password = os.getenv("NEO4J_PASSWORD")
    return GraphDatabase.driver(uri, auth=(user, password))

def setup_neo4j_constraints(driver):
    """Adds unique constraints to the Neo4j database to prevent duplicate nodes."""
    with driver.session(database="neo4j") as session:
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (c:Candidate) REQUIRE c.candidateId IS UNIQUE")
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (c:Company) REQUIRE c.name IS UNIQUE")
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (r:Role) REQUIRE r.title IS UNIQUE")
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (s:Skill) REQUIRE s.name IS UNIQUE")
    print("Neo4j constraints are set.")

def create_vector_index(driver, embedding_model):
    """Creates a vector index on the Chunk nodes for fast similarity search."""
    # Get the embedding dimension from the model
    try:
        sample_embedding = embedding_model.embed_query("sample")
        vector_dimensions = len(sample_embedding)
    except Exception as e:
        print(f"Could not determine embedding dimensions. Using default 1536. Error: {e}")
        vector_dimensions = 1536 # Default for OpenAI ada-002

    index_query = f"""
    CREATE VECTOR INDEX `project_chunks` IF NOT EXISTS
    FOR (c:Chunk) ON (c.embedding)
    OPTIONS {{ indexConfig: {{
        `vector.dimensions`: {vector_dimensions},
        `vector.similarity_function`: 'cosine'
    }} }}
    """
    with driver.session(database="neo4j") as session:
        session.run(index_query)
    print("Vector index on Chunk nodes is created.")

#### Data Ingestion and Embedding

The `ingest_data` function is responsible for populating our Neo4j database. It iterates through each candidate in our dataset and performs a three-step process:

1.  **Create Graph Structure**: It uses a `MERGE` Cypher query to create the core graph structure. `MERGE` is an idempotent operation that either finds an existing node or creates a new one, preventing duplicates. It creates nodes for the `Candidate`, `Company`, `Role`, and `Skill`, and then creates the relationships between them (e.g., `(Candidate)-[:WORKED_AT]->(Company)`).
2.  **Chunk and Embed Projects**: The unstructured `projects` text is too long for effective semantic search. We use a `RecursiveCharacterTextSplitter` to break it down into smaller, meaningful chunks. Each of these chunks is then converted into a numerical vector (an "embedding") using the OpenAI embedding model.
3.  **Store Chunks and Link to Candidate**: For each chunk, a new `:Chunk` node is created in the graph. This node stores the original text and its corresponding vector embedding. A `[:PART_OF]` relationship is created to link each chunk back to the candidate it belongs to.

In [ ]:
def ingest_data(driver, data, embedding_model, text_splitter):
    """
    Ingests candidate data into Neo4j, creates nodes, relationships,
    chunks project text, generates embeddings, and stores them.
    """
    with driver.session(database="neo4j") as session:
        for item in data:
            # 1. Create base graph (Candidate, Company, Role, Skills)
            session.run("""
                MERGE (can:Candidate {candidateId: $candidate_id})
                ON CREATE SET can.name = $name
                MERGE (com:Company {name: $company})
                MERGE (rol:Role {title: $title})
                MERGE (can)-[:WORKED_AT]->(com)
                MERGE (can)-[:HAS_ROLE]->(rol)
                
                // Merge skills and connect them to the candidate
                WITH can
                UNWIND $skills AS skill_name
                MERGE (s:Skill {name: skill_name})
                MERGE (can)-[:HAS_SKILL]->(s)
            """, **item)

            # 2. Chunk and embed the projects text
            chunks = text_splitter.split_text(item["projects"])
            embeddings = embedding_model.embed_documents(chunks)

            # 3. Create Chunk nodes and link them to the Candidate
            for i, chunk in enumerate(chunks):
                session.run("""
                    MATCH (can:Candidate {candidateId: $candidate_id})
                    CREATE (chu:Chunk {text: $text, embedding: $embedding})
                    CREATE (chu)-[:PART_OF]->(can)
                """, candidate_id=item['candidate_id'], text=chunk, embedding=embeddings[i])
            
            print(f"Ingested and embedded data for {item['name']}.")

    print("\n All data has been successfully ingested into Neo4j!")

## Main execution

In [9]:
# Initialize models and splitters
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=20)
chat_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

 # Get Neo4j driver
neo4j_driver = get_neo4j_driver()

In [10]:
 # ---- Run Ingestion (You only need to do this once) ----
setup_neo4j_constraints(neo4j_driver)
ingest_data(neo4j_driver, candidate_data, embeddings, text_splitter)
create_vector_index(neo4j_driver, embeddings)

Neo4j constraints are set.
Ingested and embedded data for Aanya Sharma.
Ingested and embedded data for Miguel Santos.
Ingested and embedded data for Priya Nair.
Ingested and embedded data for Chen Wei.
Ingested and embedded data for Sofia Rossi.
Ingested and embedded data for David Kim.

 All data has been successfully ingested into Neo4j!
Vector index on Chunk nodes is created.


### Generation Stage

We start by defining the user's query. The query has two main components:
- `role_description`: An unstructured description of the ideal candidate's experience. This will be used for the semantic search.
- `working_for_company`: A structured filter. We only want to see candidates who have worked for "ShopX".

In [11]:
# --- Performing Graph RAG Query ---

role_description = "Engineers who can build scalable cloud infrastructure on AWS"
working_for_company = "ShopX"

user_query = f"Suggest candidates who are a good fit for a role with the description : {role_description}"

# You can use am llm to extract the role description and company from the user query
query = {
    "role_description": role_description,
    "company": working_for_company
}

# creating embedding for the role_description to run semantic search
query_embedding = embeddings.embed_query(query['role_description'])

#### Step 1 : Semantic Search


This first stage performs a pure vector similarity search. The Cypher query uses the `db.index.vector.queryNodes` procedure to find the top 5 `Chunk` nodes whose embeddings are most similar to our `query_embedding`.

The query returns the `candidateId` of the candidate associated with the chunk, the similarity `score`, and the `evidence` (the text of the chunk itself). This gives us an initial list of potentially relevant candidates based on their project experience.

In [12]:
# This query finds the most similar chunks to the role description.
stage_1_semantic_query = """
    CALL db.index.vector.queryNodes('project_chunks', 5, $query_embedding) YIELD node AS chunk, score
    MATCH (chunk)-[:PART_OF]->(candidate:Candidate)
    RETURN
        candidate.candidateId AS candidateId,
        score,
        chunk.text AS evidence
    ORDER BY score DESC
"""

with neo4j_driver.session(database="neo4j") as session:
    result = session.run(stage_1_semantic_query, query_embedding=query_embedding)
    semantic_results = [record.data() for record in result]

if not semantic_results:
    print("No relevant project chunks found in the initial semantic search.")

pprint.pprint(semantic_results)

[{'candidateId': 'C006',
  'evidence': '- Deployed and managed production infrastructure on AWS using '
              'Terraform and Ansible.\n'
              '        - Implemented a monitoring and alerting system with '
              'Prometheus and Grafana for critical services.',
  'score': 0.7499716281890869},
 {'candidateId': 'C002',
  'evidence': '- Built multi-tenant EKS with IRSA and auto-scaling.\n'
              '        - Developed Python automation scripts for CI/CD '
              'pipelines, reducing deployment times by 20%.',
  'score': 0.7456197738647461},
 {'candidateId': 'C002',
  'evidence': '- Wrote technical design docs and reviewed PRs, mentoring '
              'junior engineers on best practices.\n'
              '        - Partnered with stakeholders to translate requirements '
              'into shippable increments with clear success criteria.',
  'score': 0.6874833106994629},
 {'candidateId': 'C001',
  'evidence': '- Applied RAG, Neo4j, LangGraph to delive

#### Step 2:  Enrich Context from Graph Traversal

This is where the "Graph" in GraphRAG comes into play. We take the semantically similar results from Step 1 and use the graph to both **filter** and **augment** them.

This Cypher query does the following:
1.  **`UNWIND $candidate_data AS data`**: It takes the list of candidates from our first stage as an input parameter.
2.  **`MATCH (c:Candidate ...)`**: It finds the `Candidate` nodes corresponding to the results.
3.  **`WHERE (c)-[:WORKED_AT]->(:Company {name: $company_name})`**: This is our **graph-based filter**. It traverses the `WORKED_AT` relationship to check if the candidate has worked at the specified company ("ShopX"). Candidates who do not match this pattern are discarded, even if their projects were a good semantic match.
4.  **`CALL { ... RETURN COLLECT(s.name) AS skills }`**: This subquery is our **graph-based augmentation**. For the candidates who passed the filter, it follows the `HAS_SKILL` relationships to collect a list of all their skills.
5.  **`RETURN ...`**: The final result combines the candidate's name, the original semantic score, the project evidence, and the list of skills retrieved from the graph. This creates a rich, interconnected context to pass to the LLM.

In [13]:
stage_2_filter_and_augment_query = """
    // 1. Unpack the candidate data passed in from Stage 1
    UNWIND $candidate_data AS data
    
    // 2. Match the candidate using their unique ID
    MATCH (c:Candidate {candidateId: data.candidateId})
    
    // 3. Filter: only keep candidates who worked at the specified company
    WHERE (c)-[:WORKED_AT]->(:Company {name: $company_name})
    
    // 4. Augment: For the filtered candidates, collect their skills in a subquery (UPDATED SYNTAX)
    CALL (c) {
        MATCH (c)-[:HAS_SKILL]->(s:Skill)
        RETURN COLLECT(s.name) AS skills
    }
    
    // 5. Return the final, combined context
    RETURN
        c.name AS name,
        data.score AS score,
        data.evidence AS evidence,
        skills
    ORDER BY score DESC
"""

with neo4j_driver.session(database="neo4j") as session:
    result = session.run(
        stage_2_filter_and_augment_query, 
        candidate_data = semantic_results, 
        company_name = query['company']
    )
    final_context = [record.data() for record in result]

In [14]:
final_context

[{'name': 'Miguel Santos',
  'score': 0.7456197738647461,
  'evidence': '- Built multi-tenant EKS with IRSA and auto-scaling.\n        - Developed Python automation scripts for CI/CD pipelines, reducing deployment times by 20%.',
  'skills': ['Python', 'AWS', 'EKS', 'Terraform', 'Kubernetes', 'CI/CD']},
 {'name': 'Miguel Santos',
  'score': 0.6874833106994629,
  'evidence': '- Wrote technical design docs and reviewed PRs, mentoring junior engineers on best practices.\n        - Partnered with stakeholders to translate requirements into shippable increments with clear success criteria.',
  'skills': ['Python', 'AWS', 'EKS', 'Terraform', 'Kubernetes', 'CI/CD']},
 {'name': 'Chen Wei',
  'score': 0.6324481964111328,
  'evidence': '- Optimized web performance, achieving a 40 percentage point reduction in Largest Contentful Paint (LCP).',
  'skills': ['JavaScript', 'TypeScript', 'React', 'Node.js', 'Jest']}]

#### Step 3: Generate Answer with LLM

Now that we have a highly relevant and enriched context from our graph, we can pass it to an LLM to generate a final answer.

In [15]:
prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
                You are an expert recruitment assistant. Your task is to analyze the provided candidate data
                and suggest the best fit for a role based on a user's query.
                - Use only the provided context. Do not make up information.
                - Summarize why each candidate is a good fit, citing the specific 'evidence' from their projects.
                - List the total number of unique candidates found.
                - Present the result in a clear, professional format.
                """
            ),
            (
                "human",
                """
                User Query:
                {user_query}

                Retrieved Context:
                {context}
                
                Please provide your analysis.
                """
            ),
        ]
    )

In [ ]:
# define the RAG pipeline
chain = prompt | chat_llm
response = chain.invoke({
    "user_query": user_query,
    "context": final_context
})

response 

AIMessage(content="**Candidate Analysis for Cloud Infrastructure Engineer Role on AWS**\n\n**Total Unique Candidates Found: 3**\n\n1. **Miguel Santos**\n   - **Score:** 0.7456\n   - **Evidence:**\n     - Built a multi-tenant EKS (Elastic Kubernetes Service) with IRSA (IAM Roles for Service Accounts) and auto-scaling, demonstrating his capability to design and implement scalable cloud infrastructure.\n     - Developed Python automation scripts for CI/CD pipelines, which reduced deployment times by 20%, indicating his proficiency in optimizing cloud operations and enhancing efficiency.\n   - **Skills:** Python, AWS, EKS, Terraform, Kubernetes, CI/CD\n   - **Fit Summary:** Miguel's experience with EKS and automation in AWS environments makes him a strong candidate for building scalable cloud infrastructure.\n\n2. **Miguel Santos (Second Entry)**\n   - **Score:** 0.6875\n   - **Evidence:**\n     - Wrote technical design documents and reviewed pull requests, mentoring junior engineers on be

In [ ]:
# display AI response
display(Markdown(response.content))

**Candidate Analysis for Cloud Infrastructure Engineer Role on AWS**

**Total Unique Candidates Found: 3**

1. **Miguel Santos**
   - **Score:** 0.7456
   - **Evidence:**
     - Built a multi-tenant EKS (Elastic Kubernetes Service) with IRSA (IAM Roles for Service Accounts) and auto-scaling, demonstrating his capability to design and implement scalable cloud infrastructure.
     - Developed Python automation scripts for CI/CD pipelines, which reduced deployment times by 20%, indicating his proficiency in optimizing cloud operations and enhancing efficiency.
   - **Skills:** Python, AWS, EKS, Terraform, Kubernetes, CI/CD
   - **Fit Summary:** Miguel's experience with EKS and automation in AWS environments makes him a strong candidate for building scalable cloud infrastructure.

2. **Miguel Santos (Second Entry)**
   - **Score:** 0.6875
   - **Evidence:**
     - Wrote technical design documents and reviewed pull requests, mentoring junior engineers on best practices, showcasing his leadership and collaborative skills in engineering projects.
     - Partnered with stakeholders to translate requirements into shippable increments, which reflects his ability to work effectively in a team and ensure project alignment with business goals.
   - **Skills:** Python, AWS, EKS, Terraform, Kubernetes, CI/CD
   - **Fit Summary:** This entry further emphasizes Miguel's strong background in AWS and his ability to mentor and lead projects, reinforcing his suitability for the role.

3. **Chen Wei**
   - **Score:** 0.6324
   - **Evidence:**
     - Optimized web performance, achieving a 40 percentage point reduction in Largest Contentful Paint (LCP), which indicates a focus on performance but does not directly relate to building scalable cloud infrastructure.
   - **Skills:** JavaScript, TypeScript, React, Node.js, Jest
   - **Fit Summary:** While Chen has strong skills in web performance optimization, his experience does not align closely with the requirements for building scalable cloud infrastructure on AWS.

**Recommendation:**
Based on the analysis, **Miguel Santos** is the best fit for the role, with two strong entries highlighting his relevant experience in AWS and scalable infrastructure. Chen Wei, while skilled, does not meet the specific requirements for this role.